In [2]:
# Install mlxtend
#!pip install mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
# prompt: read .csv file

import pandas as pd
transaction = pd.read_csv('/content/drive/MyDrive/data/Teaching/Transaction.csv', header=None) # Replace your_file.csv with your file name
print(transaction.head())


           0       1          2         3    4
0   keyboard   mouse  headphone       NaN  NaN
1   keyboard  SDcard      mouse       NaN  NaN
2  headphone     NaN        NaN       NaN  NaN
3   keyboard     NaN        NaN       NaN  NaN
4   keyboard  SDcard      mouse  USBdrive  NaN


In [6]:
# Step 1: One-hot encode transactions
te = TransactionEncoder()

sessions = transaction.values.tolist() #This converts the rows to a list of lists
sessions = [[str(item) for item in row] for row in sessions] # Convert all elements to strings before encoding

te_ary = te.fit(sessions).transform(sessions)
df = pd.DataFrame(te_ary, columns=te.columns_)
df

,SDcard,USBdrive,headphone,keyboard,mouse,nan
0,False,False,True,True,True,True
1,True,False,False,True,True,True
2,False,False,True,False,False,True
3,False,False,False,True,False,True
4,True,True,False,True,True,True
...,...,...,...,...,...,...
95,False,False,False,False,True,True
96,False,False,False,False,True,True
97,True,True,True,True,False,True
98,True,False,True,True,False,True


In [7]:
if 'nan' in df.columns:
    df = df.drop(columns=['nan'])
df

,SDcard,USBdrive,headphone,keyboard,mouse
0,False,False,True,True,True
1,True,False,False,True,True
2,False,False,True,False,False
3,False,False,False,True,False
4,True,True,False,True,True
...,...,...,...,...,...
95,False,False,False,False,True
96,False,False,False,False,True
97,True,True,True,True,False
98,True,False,True,True,False


In [8]:
# Step 2: Run Apriori
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)

In [9]:
frequent_itemsets

,support,itemsets
0,0.62,(SDcard)
1,0.32,(headphone)
2,0.67,(keyboard)
3,0.49,(mouse)
4,0.55,"(SDcard, keyboard)"
5,0.26,"(mouse, SDcard)"
6,0.29,"(mouse, keyboard)"
7,0.23,"(mouse, SDcard, keyboard)"


In [10]:
# Step 3: Generate rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(SDcard),(keyboard),0.62,0.67,0.55,0.887097,1.324025,1.0,0.1346,2.922857,0.644019,0.743243,0.657869,0.853996
1,(keyboard),(SDcard),0.67,0.62,0.55,0.820896,1.324025,1.0,0.1346,2.121667,0.741598,0.743243,0.528672,0.853996
2,"(mouse, SDcard)",(keyboard),0.26,0.67,0.23,0.884615,1.320321,1.0,0.0558,2.860000,0.327850,0.328571,0.650350,0.613949
3,"(mouse, keyboard)",(SDcard),0.29,0.62,0.23,0.793103,1.279199,1.0,0.0502,1.836667,0.307410,0.338235,0.455535,0.582036
4,(SDcard),"(mouse, keyboard)",0.62,0.29,0.23,0.370968,1.279199,1.0,0.0502,1.128718,0.574371,0.338235,0.114039,0.582036
5,(keyboard),"(mouse, SDcard)",0.67,0.26,0.23,0.343284,1.320321,1.0,0.0558,1.126818,0.735178,0.328571,0.112545,0.613949


In [11]:
# Step 4: Show only rules that lead to CallToAction
#cta_rules = rules[rules['consequents'].astype(str).str.contains('b', 'c')]
cta_rules = rules[rules['antecedents'].apply(lambda x: {'SDcard', 'mouse'}.issubset(x))]

cta_rules.sort_values(by='lift', ascending=False)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,"(mouse, SDcard)",(keyboard),0.26,0.67,0.23,0.884615,1.320321,1.0,0.0558,2.86,0.32785,0.328571,0.65035,0.613949


In [12]:
# Select desired columns (e.g., 'antecedents', 'consequents', 'support', 'confidence', 'lift')
filtered_rules = cta_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']]

# Display the filtered rules
filtered_rules.sort_values(by='lift', ascending=False)

,antecedents,consequents,support,confidence,lift,conviction
2,"(mouse, SDcard)",(keyboard),0.23,0.884615,1.320321,2.86


In [13]:
cta_rules = rules[rules['consequents'].astype(str).str.contains('SDcard')]
cta_rules.sort_values(by='lift', ascending=False)

filtered_rules = cta_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']]
filtered_rules.sort_values(by='lift', ascending=False)

,antecedents,consequents,support,confidence,lift,conviction
1,(keyboard),(SDcard),0.55,0.820896,1.324025,2.121667
5,(keyboard),"(mouse, SDcard)",0.23,0.343284,1.320321,1.126818
3,"(mouse, keyboard)",(SDcard),0.23,0.793103,1.279199,1.836667
